In [ ]:
# Install necessary libraries
!pip install modelscope==1.4.2
!pip install open_clip_torch
!pip install pytorch-lightning
!pip install numpy
!pip install scikit-image
!pip install torchvision
!pip install opencv-python

# Import necessary libraries
from huggingface_hub import snapshot_download
from modelscope.pipelines import pipeline
from modelscope.outputs import OutputKeys
import pathlib
import logging
import numpy as np
from skimage.metrics import structural_similarity as ssim, peak_signal_noise_ratio as psnr
import cv2
import time

  Using cached open_clip_torch-2.24.0-py3-none-any.whl (1.5 MB)
  Using cached ftfy-6.2.0-py3-none-any.whl (54 kB)
  Using cached timm-1.0.7-py3-none-any.whl (2.3 MB)
  Using cached nvidia_cuda_nvrtc_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (23.7 MB)
  Using cached nvidia_cuda_runtime_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (823 kB)
  Using cached nvidia_cuda_cupti_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (14.1 MB)
  Using cached nvidia_cudnn_cu12-8.9.2.26-py3-none-manylinux1_x86_64.whl (731.7 MB)
  Using cached nvidia_cublas_cu12-12.1.3.1-py3-none-manylinux1_x86_64.whl (410.6 MB)
  Using cached nvidia_cufft_cu12-11.0.2.54-py3-none-manylinux1_x86_64.whl (121.6 MB)
  Using cached nvidia_curand_cu12-10.3.2.106-py3-none-manylinux1_x86_64.whl (56.5 MB)
  Using cached nvidia_cusolver_cu12-11.4.5.107-py3-none-manylinux1_x86_64.whl (124.2 MB)
  Using cached nvidia_cusparse_cu12-12.1.0.106-py3-none-manylinux1_x86_64.whl (196.0 MB)
  Using cached nvidia_nccl_cu12-2.20.5-py3-none-ma

2024-07-03 18:31:22,311 - modelscope - INFO - PyTorch version 2.3.0+cu121 Found.
2024-07-03 18:31:22,317 - modelscope - INFO - TensorFlow version 2.15.0 Found.
2024-07-03 18:31:22,320 - modelscope - INFO - Loading ast index from /root/.cache/modelscope/ast_indexer
2024-07-03 18:31:22,324 - modelscope - INFO - No valid ast index found from /root/.cache/modelscope/ast_indexer, generating ast index from prebuilt!
2024-07-03 18:31:22,390 - modelscope - INFO - Loading done! Current index file version is 1.4.2, with md5 3b1ba307477be50d5f16e0127bda71f4 and a total number of 842 components indexed


In [ ]:
# Set up logging
logging.basicConfig(level=logging.INFO)

# Define model directory
model_dir = pathlib.Path('weights')

# Download the model
try:
    logging.info("Downloading model...")
    snapshot_download('damo-vilab/modelscope-damo-text-to-video-synthesis', repo_type='model', local_dir=model_dir)
    logging.info("Model downloaded successfully.")
except Exception as e:
    logging.error(f"Error downloading model: {e}")
    raise

# Load the pipeline
try:
    logging.info("Loading pipeline...")
    pipe = pipeline('text-to-video-synthesis', model_dir.as_posix())
    logging.info("Pipeline loaded successfully.")
except Exception as e:
    logging.error(f"Error loading pipeline: {e}")
    raise

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

configuration.json:   0%|          | 0.00/1.07k [00:00<?, ?B/s]

README.md:   0%|          | 0.00/5.49k [00:00<?, ?B/s]

.gitattributes:   0%|          | 0.00/1.48k [00:00<?, ?B/s]

...

VQGAN_autoencoder.pth:   0%|          | 0.00/5.21G [00:00<?, ?B/s]

2024-07-03 18:36:58,269 - modelscope - INFO - initiate model from weights
2024-07-03 18:36:58,271 - modelscope - INFO - initiate model from location weights.
2024-07-03 18:36:58,276 - modelscope - INFO - initialize model from weights
The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

ERROR:root:Error loading pipeline: TextToVideoSynthesisPipeline: TextToVideoSynthesis: Attempting to deserialize object on a CUDA device but torch.cuda.is_available() is False. If you are running on a CPU-only machine, please use torch.load with map_location=torch.device('cpu') to map your storages to the CPU.


RuntimeError: TextToVideoSynthesisPipeline: TextToVideoSynthesis: Attempting to deserialize object on a CUDA device but torch.cuda.is_available() is False. If you are running on a CPU-only machine, please use torch.load with map_location=torch.device('cpu') to map your storages to the CPU.

In [ ]:
# Function to calculate SSIM and PSNR
def calculate_metrics(video_path):
    try:
        cap = cv2.VideoCapture(video_path)
        if not cap.isOpened():
            logging.error(f"Error opening video file: {video_path}")
            return None, None

        ssim_values = []
        psnr_values = []

        ret, ref_frame = cap.read()
        if not ret:
            logging.error(f"Error reading video frame: {video_path}")
            return None, None

        ref_frame_gray = cv2.cvtColor(ref_frame, cv2.COLOR_BGR2GRAY)

        while True:
            ret, frame = cap.read()
            if not ret:
                break
            frame_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            ssim_values.append(ssim(ref_frame_gray, frame_gray))
            psnr_values.append(psnr(ref_frame_gray, frame_gray))

        cap.release()

        avg_ssim = np.mean(ssim_values)
        avg_psnr = np.mean(psnr_values)

        return avg_ssim, avg_psnr
    except Exception as e:
        logging.error(f"Error calculating metrics: {e}")
        return None, None


In [ ]:
# Define test texts
test_texts = [
    {'text': 'A man without a job.'},
    #{'text': 'A cat playing with a ball of yarn.'},
    #{'text': 'A sunset over a mountain range.'}
]

In [ ]:
# Generate videos and collect metrics
metrics = []

for test_text in test_texts:
    try:
        logging.info(f"Generating video for text: {test_text['text']}")
        start_time = time.time()
        output = pipe(test_text)
        if OutputKeys.OUTPUT_VIDEO not in output:
            logging.error("Output video not found in pipeline output.")
            continue
        output_video_path = output[OutputKeys.OUTPUT_VIDEO]
        end_time = time.time()
        generation_time = end_time - start_time
        logging.info(f"Video generated successfully: {output_video_path}")

        avg_ssim, avg_psnr = calculate_metrics(output_video_path)
        if avg_ssim is None or avg_psnr is None:
            logging.warning(f"Metrics calculation failed for video: {output_video_path}")
            continue

        metrics.append({
            'text': test_text['text'],
            'video_path': output_video_path,
            'ssim': avg_ssim,
            'psnr': avg_psnr,
            'generation_time': generation_time
        })
    except Exception as e:
        logging.error(f"Error generating video: {e}")

2024-06-30 14:26:52,837 - modelscope - WARNING - task text-to-video-synthesis input definition is missing
2024-06-30 14:27:57,984 - modelscope - WARNING - task text-to-video-synthesis output keys are missing


In [ ]:
# Download result
from google.colab import files
files.download(output_video_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# Print collected metrics
for metric in metrics:
    print(f"Text: {metric['text']}")
    print(f"Video Path: {metric['video_path']}")
    print(f"SSIM: {metric['ssim']}")
    print(f"PSNR: {metric['psnr']}")
    print(f"Generation Time: {metric['generation_time']} seconds\n")

Text: A panda eating bamboo on a rock.
Video Path: /tmp/tmpe28pjyt5.mp4
SSIM: 0.24690402161099603
PSNR: 12.681235431543055
Generation Time: 65.14930987358093 seconds

